# DSA 210 DATA SCIENCE PROJECT 
## The Impact of Crude Oil Prices on Food Sector Stock Prices


---
Alperen SARIŞEN 33886


## Part 1. Imports & Setup

In this initial step, we import the necessary Python libraries for data manipulation (pandas), visualization (matplotlib, seaborn), and statistical analysis (scipy). We then fetch historical stock data for the food sector (KO, KHC, MCD, MDLZ) via yfinance and load the daily Brent Crude Oil price dataset. Both datasets are merged on a daily date key, and missing values—primarily due to stock market holidays and weekends—are handled using the forward-filling method to ensure a continuous time series for analysis.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.stats import pearsonr, spearmanr, shapiro, ttest_ind



## 2. Data Collection & Loading

We use:
- **Brent Oil Prices** – downloaded from Kaggle (`BrentOilPrices.csv`)
- **Food Sector Stocks** – downloaded via `yfinance`: KO, KHC, MCD, MDLZ


In [14]:
# Load brent oil from CSV (downloaded from Kaggle)
brent = pd.read_csv("BrentOilPrices.csv")
brent = brent.rename(columns={"Date": "date", "Price": "brent_price"})
brent["date"] = pd.to_datetime(brent["date"], dayfirst=True)
brent = brent.sort_values("date")
brent = brent[(brent["date"] >= "2015-01-01") & (brent["date"] <= "2023-12-31")]
brent = brent.reset_index(drop=True)

print("Brent oil loaded:", len(brent), "rows")
print(brent["date"].min(), "->", brent["date"].max())

Brent oil loaded: 2005 rows
2015-01-02 00:00:00 -> 2022-11-14 00:00:00


In [15]:
# Download stock data using yfinance
KO   = yf.download("KO",   start="2015-01-01", end="2023-12-31", auto_adjust=True)["Close"]
KHC  = yf.download("KHC",  start="2015-01-01", end="2023-12-31", auto_adjust=True)["Close"]
MCD  = yf.download("MCD",  start="2015-01-01", end="2023-12-31", auto_adjust=True)["Close"]
MDLZ = yf.download("MDLZ", start="2015-01-01", end="2023-12-31", auto_adjust=True)["Close"]

print("KO shape:",   KO.shape)
print("KHC shape:",  KHC.shape)
print("MCD shape:",  MCD.shape)
print("MDLZ shape:", MDLZ.shape)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

KO shape: (2264, 1)
KHC shape: (2138, 1)
MCD shape: (2264, 1)
MDLZ shape: (2264, 1)


## 3. Merge Datasets

In [18]:
# Put stocks into one dataframe
stocks = pd.DataFrame({
    "date": KO.index,
    "KO":   KO.values.flatten(),
    "KHC":  KHC.reindex(KO.index).values.flatten(),
    "MCD":  MCD.reindex(KO.index).values.flatten(),
    "MDLZ": MDLZ.reindex(KO.index).values.flatten()
})
stocks["date"] = pd.to_datetime(stocks["date"])

# Merge with brent
df = pd.merge(brent, stocks, on="date", how="outer")
df = df.sort_values("date")
df = df.ffill()
df = df.dropna()
df = df.reset_index(drop=True)

print(df.shape)
df.head()


(2180, 6)


,date,brent_price,KO,KHC,MCD,MDLZ
0,2015-07-06,57.19,28.130497,45.202484,73.337120,32.302959
1,2015-07-07,54.72,28.722803,46.237125,74.126854,32.710083
2,2015-07-08,55.70,28.444490,46.714191,73.482803,32.420414
3,2015-07-09,57.83,28.487303,46.342472,73.981194,32.295128
4,2015-07-10,57.72,28.822701,47.897533,74.870598,32.420414
